# Javtiful → GoFile Telegram Bot

Google Colab runner for the `gofile` branch. Paste your Telegram bot token directly into **Setup** when prompted; input is hidden and the token is not saved in this notebook. Owner Telegram ID `7367490186` is preset.

### Run steps
1. Run **Setup** and paste your `BOT_TOKEN` when prompted.
2. Run **Check** to compile the bot and execute its regression tests.
3. Run **Start bot** and search a full code (such as `IPZZ-989`) or prefix (such as `IPZZ`).
4. For bulk mode, send up to 20 codes separated by new lines, commas, or semicolons. Choose **Download all**, **Screenshots all**, or **Download + Screenshots all**.
5. Use `/parallel` in Telegram to set 1–5 download workers, 1–3 GoFile upload workers, and 1–2 screenshot workers. Defaults are 3 downloads, 2 uploads, and 1 screenshot worker; changes apply to the next batch.

### Bulk pipeline behavior
- Different videos can be in progress at the same time. Each video follows its own sequence: **download → screenshots (if selected) → GoFile upload (if selected)**.
- Active video jobs are capped by the download-worker setting, and each stage has its own worker limit. This overlaps work while preventing an unbounded pile of downloaded videos from filling Colab storage.
- Successful video codes are collected in one compact Telegram status message, for example `START-618✅ START-627✅ IPZZ-950✅`, instead of sending a separate success message for each upload.
- Download and upload progress are shown separately per video, including speed and ETA when available. Failed items do not stop the rest of the batch.
- Screenshots can be sent to the current chat or private channel `-1003963567446`; add the bot to that channel with permission to post first.


In [ ]:
# Setup — enter bot token, install ffmpeg, and fetch only the gofile branch
import os
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

BOT_TOKEN = getpass("Paste your Telegram bot token (input hidden): ").strip()
if not BOT_TOKEN:
    raise RuntimeError("BOT_TOKEN cannot be empty.")
os.environ["BOT_TOKEN"] = BOT_TOKEN
del BOT_TOKEN


REPO_DIR = Path("/content/URL-uploader-gofile")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "gofile", "https://github.com/Telebotfaroff/URL-uploader.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Setup complete. Bot token input was hidden.")
print("Screenshot channel: -1003963567446 (bot must be added as an admin or have permission to post messages).")

In [ ]:
# Check — compile the required code and run the GoFile regression tests
import subprocess
import sys

subprocess.run([sys.executable, "-m", "compileall", "-q", "bot.py", "gofile_bot.py", "gofile_search.py", "gofile_media.py", "gofile_uploader.py", "plugins/javdl_core"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_gofile_bot.py", "-v"], check=True)
print("Checks passed.")

In [ ]:
# Start bot — keep this cell running while you use the bot
import os
import subprocess
import sys

if not os.environ.get("BOT_TOKEN"):
    raise RuntimeError("Run the Setup cell first and enter your bot token when prompted.")
subprocess.run([sys.executable, "gofile_bot.py"], check=True)